# Notebook 2: 运动分类器训练

训练轻量级分类器, 对每个波比跳窗口生成简单运动类型的概率分布。

**关键验证**: 波比跳分布随时间变化的热力图。

In [ ]:
import pickle
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.preprocessing import StandardScaler, LabelEncoder

PROJECT_ROOT = Path(r"D:\data\PPG_HeartRate\Algorithm\outline-PPGtoHR")
ARTIFACTS_DIR = PROJECT_ROOT / "docs" / "research" / "artifacts"
sys.path.insert(0, str(PROJECT_ROOT / "python" / "src"))

plt.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei"]
plt.rcParams["axes.unicode_minus"] = False
sns.set_style("whitegrid")

EXERCISES = {
    "jump_rope": {"label": "跳绳"},
    "arm_curl": {"label": "手臂弯举"},
    "push_up": {"label": "俯卧撑"},
    "jumping_jack": {"label": "开合跳"},
    "burpee": {"label": "波比跳"},
}

## 1. 加载特征数据

In [ ]:
with open(ARTIFACTS_DIR / "mimu_features_all.pkl", "rb") as f:
    data = pickle.load(f)

X = data["X"]
y = data["y"]
df = data["df"]
FEATURE_NAMES = data["feature_names"]

print(f"特征矩阵: {X.shape}")
print(f"运动类型: {np.unique(y)}")

## 2. 训练/测试集划分

- 训练集: 仅简单运动窗口
- 波比跳仅用于推理 (不参与训练)

In [ ]:
simple_mask = df["exercise"].values != "burpee"
burpee_mask = ~simple_mask

X_train_all = X[simple_mask]
y_train_all = y[simple_mask]
files_train_all = df["file_name"].values[simple_mask]

X_burpee = X[burpee_mask]
y_burpee = y[burpee_mask]
burpee_times = df["window_time"].values[burpee_mask]
burpee_files = df["file_name"].values[burpee_mask]

print(f"训练集 (简单运动): {X_train_all.shape[0]} 个窗口")
print(f"推理集 (波比跳): {X_burpee.shape[0]} 个窗口")
print(f"训练集类别: {dict(zip(*np.unique(y_train_all, return_counts=True)))}")

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_all)
X_burpee_scaled = scaler.transform(X_burpee)

le = LabelEncoder()
y_train_encoded = le.fit_transform(y_train_all)
print(f"类别映射: {dict(zip(le.classes_, le.transform(le.classes_)))}")
SIMPLE_CLASSES = list(le.classes_)
print(f"简单运动类别: {SIMPLE_CLASSES}")

## 3. 分类器训练和评估

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

rf = RandomForestClassifier(
    n_estimators=100, max_depth=10, min_samples_leaf=5,
    class_weight="balanced", random_state=42,
)
rf_scores = cross_val_score(rf, X_train_scaled, y_train_encoded, cv=cv, scoring="accuracy")
print(f"RandomForest CV accuracy: {rf_scores.mean():.3f} +/- {rf_scores.std():.3f}")

knn = KNeighborsClassifier(n_neighbors=5)
knn_scores = cross_val_score(knn, X_train_scaled, y_train_encoded, cv=cv, scoring="accuracy")
print(f"KNN CV accuracy: {knn_scores.mean():.3f} +/- {knn_scores.std():.3f}")

In [ ]:
rf.fit(X_train_scaled, y_train_encoded)

y_pred = rf.predict(X_train_scaled)
print("RandomForest 训练集分类报告:")
print(classification_report(y_train_encoded, y_pred, target_names=le.classes_))

In [ ]:
cm = confusion_matrix(y_train_encoded, y_pred)
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=[EXERCISES[c]["label"] for c in le.classes_],
            yticklabels=[EXERCISES[c]["label"] for c in le.classes_],
            ax=ax)
ax.set_xlabel("预测")
ax.set_ylabel("真实")
ax.set_title("混淆矩阵 - 简单运动分类")
plt.tight_layout()
plt.show()

In [ ]:
importances = rf.feature_importances_
top_k = 15
top_idx = np.argsort(importances)[-top_k:]

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(range(top_k), importances[top_idx])
ax.set_yticks(range(top_k))
ax.set_yticklabels([FEATURE_NAMES[i] for i in top_idx])
ax.set_xlabel("Feature Importance")
ax.set_title(f"Top {top_k} 特征重要性 (RandomForest)")
plt.tight_layout()
plt.show()

## 4. 波比跳窗口分布推理

In [ ]:
burpee_proba = rf.predict_proba(X_burpee_scaled)

print(f"波比跳概率分布矩阵: {burpee_proba.shape}")
print(f"类别对应: {list(zip(le.classes_, range(len(le.classes_))))}")
print(f"\n各窗口最大概率: mean={burpee_proba.max(axis=1).mean():.3f}, "
      f"min={burpee_proba.max(axis=1).min():.3f}, max={burpee_proba.max(axis=1).max():.3f}")

In [ ]:
unique_burpee_files = np.unique(burpee_files)

for bf in unique_burpee_files:
    file_mask = burpee_files == bf
    proba_file = burpee_proba[file_mask]
    times_file = burpee_times[file_mask]

    fig, ax = plt.subplots(figsize=(14, 4))
    sns.heatmap(
        proba_file.T,
        xticklabels=50,
        yticklabels=[EXERCISES[c]["label"] for c in le.classes_],
        cmap="YlOrRd", vmin=0, vmax=1, ax=ax,
    )
    ax.set_xlabel("窗口索引")
    ax.set_ylabel("运动类型")
    ax.set_title(f"波比跳窗口概率分布 - {bf}")
    plt.tight_layout()
    plt.show()

In [ ]:
for bf in unique_burpee_files:
    file_mask = burpee_files == bf
    proba_file = burpee_proba[file_mask]
    times_file = burpee_times[file_mask]

    fig, ax = plt.subplots(figsize=(14, 5))
    ax.stackplot(
        range(len(times_file)),
        *[proba_file[:, i] for i in range(proba_file.shape[1])],
        labels=[EXERCISES[c]["label"] for c in le.classes_],
        alpha=0.8,
    )
    ax.set_xlabel("窗口索引")
    ax.set_ylabel("概率")
    ax.set_title(f"波比跳运动分布随时间变化 - {bf}")
    ax.legend(loc="upper right")
    ax.set_ylim(0, 1)
    plt.tight_layout()
    plt.show()

## 5. 保存分类器输出

In [ ]:
import joblib

np.save(ARTIFACTS_DIR / "burpee_window_distributions.npy", burpee_proba)
joblib.dump({"model": rf, "scaler": scaler, "label_encoder": le},
            ARTIFACTS_DIR / "classifier_model.pkl")

burpee_meta = {
    "times": burpee_times,
    "files": burpee_files,
    "class_names": SIMPLE_CLASSES,
}
with open(ARTIFACTS_DIR / "burpee_meta.pkl", "wb") as f:
    pickle.dump(burpee_meta, f)

print("已保存:")
print(f"  burpee_window_distributions.npy: shape={burpee_proba.shape}")
print(f"  classifier_model.pkl")
print(f"  burpee_meta.pkl")